# Fine-tuning Phi-3 Mini for Text Rewriting Tasks

This notebook demonstrates how to fine-tune Microsoft's Phi-3 Mini model on a custom text rewriting dataset using QLoRA (Quantized Low-Rank Adaptation) for efficient training on Google Colab.

**What you'll learn:**
- How to prepare your dataset for instruction fine-tuning
- How to load and quantize Phi-3 Mini for memory efficiency
- How to apply LoRA adapters for parameter-efficient fine-tuning
- How to train and save your fine-tuned model
- How to run inference with the fine-tuned model

## Step 1: Install Required Libraries

We need to install the following:
- `transformers`: For loading Phi-3 Mini
- `peft`: For LoRA/QLoRA fine-tuning
- `bitsandbytes`: For 4-bit quantization
- `trl`: For supervised fine-tuning trainer
- `accelerate`: For distributed training utilities
- `datasets`: For dataset handling

In [ ]:
# Install required packages
!pip install -q transformers>=4.40.0
!pip install -q peft>=0.10.0
!pip install -q bitsandbytes>=0.43.0
!pip install -q trl>=0.8.0
!pip install -q accelerate>=0.28.0
!pip install -q datasets>=2.18.0
!pip install -q flash-attn --no-build-isolation

print("✅ All packages installed successfully!")

## Step 2: Import Libraries and Check GPU

In [ ]:
import torch
import json
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
import warnings
warnings.filterwarnings('ignore')

# Check GPU availability
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ No GPU detected! Go to Runtime > Change runtime type > GPU")

## Step 3: Prepare Your Dataset

We'll format your data for instruction fine-tuning using Phi-3's chat template.

In [ ]:
# Your training data - text rewriting examples
training_data = [
    {"input": "Rewrite to be friendlier: 'Send me the report by EOD.'", "output": "Could you please send me the report by end of day? Thanks!"},
    {"input": "Make more professional: 'Your app is broken. Fix it.'", "output": "It looks like the application isn't working as expected. Could you investigate and provide a fix?"},
    {"input": "Simplify: 'Subsequent to our prior correspondence, I am following up.'", "output": "I'm following up on our previous message."},
    {"input": "Rewrite as a short SMS: 'I will be 10 minutes late due to traffic.'", "output": "Running ~10 mins late—traffic. See you soon."},
    {"input": "Fix grammar: 'Me and Alex were going to the store yesterday.'", "output": "Alex and I went to the store yesterday."},
    {"input": "Make less harsh: 'This is wrong. Do it again.'", "output": "I think there's an issue with this—could you revise it and try again?"},
    {"input": "Rewrite to be assertive but polite: 'If you can, maybe send the file?'", "output": "Please send the file when you get a chance."},
    {"input": "Turn into an email subject: 'Request for invoice for November 2025'", "output": "Invoice Request — November 2025"},
    {"input": "Rewrite in 2 sentences for a cover letter: 'I like data and I want the job.'", "output": "I enjoy turning data into clear insights that drive decisions. I'm excited to bring that skill and a strong ownership mindset to this role."},
    {"input": "Shorten: 'At this point in time we are unable to proceed.'", "output": "We can't proceed right now."},
    {"input": "Rewrite to sound confident: 'I think I can do this project.'", "output": "I'm confident I can deliver this project successfully."},
    {"input": "Rewrite as a polite decline: 'No, I won't join the call.'", "output": "Thanks for the invite, but I won't be able to join the call."},
    {"input": "Rewrite in plain English: 'We will commence implementation following stakeholder alignment.'", "output": "We'll start implementation once everyone agrees."},
    {"input": "Rewrite as bullets: 'We need design, then build, then test.'", "output": "- Design\n- Build\n- Test"},
    {"input": "Make more empathetic: 'Read the docs.'", "output": "The docs cover this—tell me where you're stuck and I'll point you to the right section."},
    {"input": "Remove passive voice: 'The decision was made to cancel.'", "output": "We decided to cancel."},
    {"input": "Write a thank-you note: 'Thanks for your help yesterday.'", "output": "Thank you for your help yesterday—I really appreciate it."},
    {"input": "Rewrite for LinkedIn (1 sentence): 'I started a new job as a backend engineer.'", "output": "Excited to share I've started a new role as a Backend Engineer!"},
    {"input": "Rewrite to be neutral: 'This feature is terrible.'", "output": "This feature isn't meeting our needs right now; here are the main issues to address."},
    {"input": "Rewrite as meeting invite title: 'Let's talk about the roadmap next week.'", "output": "Roadmap Discussion — Next Week"},
    {"input": "Make concise: 'Please be advised that your request has been received.'", "output": "We received your request."},
    {"input": "Rewrite politely: 'Stop pinging me.'", "output": "I saw your messages—I'll respond as soon as I can."},
    {"input": "Rewrite clearer: 'We need that thing done ASAP.'", "output": "Please complete this as soon as possible. Let me know your ETA."},
    {"input": "Rewrite to set boundary: 'I can do it tonight.'", "output": "I can start tonight, but I may need until tomorrow to finish."},
    {"input": "Rewrite as apology: 'I missed your email.'", "output": "Sorry I missed your email—thanks for following up."},
    {"input": "Summarize in 1 sentence: 'We shipped v2 yesterday. Load times improved, but some older Android devices have login errors. A hotfix is planned tomorrow.'", "output": "v2 improved load times, but older Android devices see login errors and a hotfix is planned for tomorrow."},
    {"input": "Summarize in 3 bullets: 'DB migration this weekend. Expect 2 hours read-only. Support should post a status update and monitor tickets.'", "output": "- Database migration this weekend\n- About 2 hours in read-only mode\n- Support posts a status update and monitors tickets"},
    {"input": "Give a TL;DR: 'Q4 revenue grew 8% YoY from enterprise renewals. SMB churn increased. Q1 focus: onboarding and pricing tests.'", "output": "TL;DR: Q4 grew 8% YoY via enterprise renewals, SMB churn rose, and Q1 will focus on onboarding and pricing tests."},
    {"input": "Summarize for a manager: 'Engineering is blocked because the API contract keeps changing. We need a locked spec and weekly sync.'", "output": "Engineering is blocked by changing API requirements; we need a locked spec and a weekly sync to reduce rework."},
    {"input": "Summarize as action list: 'Safari 15 bug reports; repro unclear. Ask QA for screen capture and console logs.'", "output": "- Ask QA to record a screen capture\n- Collect Safari 15 console logs\n- Update the bug with clear repro steps"},
    {"input": "Summarize in one line: 'Training failed because the dataset has invalid JSON on line 57.'", "output": "Training failed due to invalid JSON at line 57."},
    {"input": "Write an email subject: 'Incident report for partial outage affecting EU users on Dec 10.'", "output": "Dec 10 EU Partial Outage — Incident Report"},
    {"input": "Summarize: 'Deprecate v1 endpoints. Keep for 90 days, publish migration guide, add warnings.'", "output": "Deprecate v1 endpoints by keeping them for 90 days, publishing a migration guide, and adding response warnings."},
    {"input": "Summarize in 2 sentences: 'Variant B reduced checkout abandonment by 4% and slightly lowered AOV. Revenue per visitor increased 1.2% overall.'", "output": "Variant B reduced abandonment by 4% with a slight AOV drop, but increased revenue per visitor by 1.2% overall."},
    {"input": "Summarize as a title: 'Proposal to reduce costs by switching object storage providers.'", "output": "Proposal: Reduce Object Storage Costs by Switching Providers"},
    {"input": "Summarize in 3 bullets: 'Confirm requirements, draft spec, review, implement, QA.'", "output": "- Confirm requirements and draft spec\n- Review with stakeholders\n- Implement and run QA"},
    {"input": "Summarize: 'Customer wants SSO by March plus audit logs and a dedicated Slack channel; they'll sign a 2-year deal if we commit.'", "output": "Customer will sign a 2-year deal if we commit to SSO by March, audit logs, and a dedicated Slack channel."},
    {"input": "Summarize: 'We paused the rollout after error rates rose. We're investigating and will update at 3pm.'", "output": "Rollout is paused due to increased errors; investigation is underway with an update planned at 3pm."},
    {"input": "Summarize: 'The meeting agreed to move the deadline from Friday to next Wednesday because QA needs more time.'", "output": "They moved the deadline from Friday to next Wednesday to give QA more time."},
    {"input": "Summarize: 'We should add caching, reduce payload size, and index the DB table to improve latency.'", "output": "To improve latency, add caching, reduce payload size, and add a database index."},
    {"input": "Extract the due date: 'Please deliver the proposal by 2025-12-20.'", "output": "2025-12-20"},
    {"input": "Extract names: 'Participants: Maya Chen, Omar Ali, Priya Singh.'", "output": "Maya Chen; Omar Ali; Priya Singh"},
    {"input": "Convert to JSON: name=Rita, role=Analyst, active=true", "output": "{\"name\":\"Rita\",\"role\":\"Analyst\",\"active\":true}"},
    {"input": "Normalize (lowercase, trim): '  Apple  , BANANA,  kiwi '", "output": "apple, banana, kiwi"},
    {"input": "Extract the email: 'Contact dev.team+alerts@example.com for urgent issues.'", "output": "dev.team+alerts@example.com"},
    {"input": "Convert to ISO date: 'Dec 16, 2025'", "output": "2025-12-16"},
    {"input": "Extract numbers: '12 errors Monday, 7 Tuesday, 0 Wednesday.'", "output": "12, 7, 0"}
]

print(f"📊 Total training examples: {len(training_data)}")

In [ ]:
# Format data for Phi-3's chat template
# Phi-3 uses the ChatML format: <|user|>\n{message}<|end|>\n<|assistant|>\n{response}<|end|>

def format_instruction(sample):
    """Format a single example into Phi-3's chat format."""
    system_message = "You are a helpful text rewriting assistant. Transform the input text as instructed."
    
    # Phi-3 chat format
    formatted = f"""<|system|>
{system_message}<|end|>
<|user|>
{sample['input']}<|end|>
<|assistant|>
{sample['output']}<|end|>"""
    
    return formatted

# Create formatted dataset
formatted_data = [{"text": format_instruction(sample)} for sample in training_data]

# Convert to HuggingFace Dataset
dataset = Dataset.from_list(formatted_data)

print("\n📝 Example formatted prompt:")
print("=" * 50)
print(formatted_data[0]["text"])
print("=" * 50)

## Step 4: Load Phi-3 Mini with 4-bit Quantization

We use QLoRA (4-bit quantization + LoRA) to fit the model in Colab's GPU memory.

In [ ]:
# Model configuration
model_id = "microsoft/Phi-3-mini-4k-instruct"

# 4-bit quantization config for memory efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                      # Load model in 4-bit precision
    bnb_4bit_quant_type="nf4",              # Use NormalFloat4 quantization
    bnb_4bit_compute_dtype=torch.bfloat16,  # Compute in bfloat16 for stability
    bnb_4bit_use_double_quant=True,         # Nested quantization for more memory savings
)

print(f"🚀 Loading model: {model_id}")
print("This may take a few minutes...")

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    trust_remote_code=True,
    use_fast=True
)

# Set padding token (Phi-3 doesn't have one by default)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # Pad on the right for causal LM

print(f"✅ Tokenizer loaded")
print(f"   Vocab size: {tokenizer.vocab_size}")
print(f"   Pad token: {tokenizer.pad_token}")

In [ ]:
# Load model with quantization
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",           # Automatically place on GPU
    trust_remote_code=True,
    attn_implementation="eager",  # Use eager attention (flash-attn optional)
)

# Disable caching for training
model.config.use_cache = False
model.config.pretraining_tp = 1

print(f"\n✅ Model loaded successfully!")
print(f"   Model size: ~{model.get_memory_footprint() / 1e9:.2f} GB in memory")

## Step 5: Configure LoRA Adapters

LoRA (Low-Rank Adaptation) allows us to fine-tune only a small number of parameters instead of the full model.

In [ ]:
# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)

# LoRA configuration
lora_config = LoraConfig(
    r=16,                          # Rank of the low-rank matrices
    lora_alpha=32,                 # Scaling factor (alpha/r is the actual scaling)
    lora_dropout=0.05,             # Dropout for regularization
    bias="none",                   # Don't train bias terms
    task_type="CAUSAL_LM",         # Task type for causal language modeling
    target_modules=[               # Which layers to apply LoRA to
        "q_proj",                  # Query projection
        "k_proj",                  # Key projection
        "v_proj",                  # Value projection
        "o_proj",                  # Output projection
        "gate_proj",               # MLP gate
        "up_proj",                 # MLP up projection
        "down_proj",               # MLP down projection
    ],
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)

# Print trainable parameters
model.print_trainable_parameters()

print("\n✅ LoRA adapters configured!")

## Step 6: Configure Training Arguments

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir="./phi3-text-rewriter",           # Where to save checkpoints
    
    # Training hyperparameters
    num_train_epochs=3,                          # Number of training epochs
    per_device_train_batch_size=2,               # Batch size per GPU
    gradient_accumulation_steps=4,               # Accumulate gradients (effective batch = 8)
    
    # Optimizer settings
    learning_rate=2e-4,                          # Learning rate
    weight_decay=0.01,                           # L2 regularization
    warmup_ratio=0.1,                            # Warmup for 10% of training
    lr_scheduler_type="cosine",                  # Cosine learning rate decay
    
    # Memory optimization
    optim="paged_adamw_8bit",                    # 8-bit Adam for memory savings
    fp16=False,                                  # Don't use fp16
    bf16=True,                                   # Use bfloat16 (better for training)
    gradient_checkpointing=True,                 # Save memory by recomputing activations
    max_grad_norm=0.3,                           # Gradient clipping
    
    # Logging and saving
    logging_steps=10,                            # Log every 10 steps
    save_strategy="epoch",                       # Save at end of each epoch
    save_total_limit=2,                          # Keep only last 2 checkpoints
    
    # Other settings
    report_to="none",                            # Disable wandb/tensorboard
    seed=42,                                     # Random seed for reproducibility
)

print("✅ Training arguments configured!")
print(f"   Epochs: {training_args.num_train_epochs}")
print(f"   Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"   Learning rate: {training_args.learning_rate}")

## Step 7: Create Trainer and Start Training

In [ ]:
# Create the SFT (Supervised Fine-Tuning) Trainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
    dataset_text_field="text",           # Column containing the formatted text
    max_seq_length=512,                  # Maximum sequence length
    packing=False,                       # Don't pack multiple examples into one sequence
)

print("✅ Trainer created!")
print(f"   Training examples: {len(dataset)}")
print(f"   Max sequence length: 512")

In [ ]:
# Start training!
print("🎯 Starting fine-tuning...")
print("=" * 50)

# Clear CUDA cache before training
torch.cuda.empty_cache()

# Train the model
trainer.train()

print("\n" + "=" * 50)
print("✅ Training complete!")

## Step 8: Save the Fine-tuned Model

In [ ]:
# Save the LoRA adapters
output_dir = "./phi3-text-rewriter-final"
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"✅ Model saved to: {output_dir}")
print("\nSaved files:")
!ls -la {output_dir}

## Step 9: Test the Fine-tuned Model

In [ ]:
# Function to generate text with the fine-tuned model
def generate_response(prompt, max_new_tokens=128):
    """Generate a response using the fine-tuned model."""
    
    # Format the prompt in Phi-3's chat format
    system_message = "You are a helpful text rewriting assistant. Transform the input text as instructed."
    formatted_prompt = f"""<|system|>
{system_message}<|end|>
<|user|>
{prompt}<|end|>
<|assistant|>
"""
    
    # Tokenize
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    # Decode and extract response
    full_response = tokenizer.decode(outputs[0], skip_special_tokens=False)
    
    # Extract just the assistant's response
    response = full_response.split("<|assistant|>")[-1].replace("<|end|>", "").strip()
    
    return response

print("✅ Inference function ready!")

In [ ]:
# Test with examples from training data (should work well)
print("📝 Testing with training examples:")
print("=" * 60)

test_prompts = [
    "Rewrite to be friendlier: 'Get this done now.'",
    "Simplify: 'In accordance with the aforementioned guidelines...'",
    "Make more professional: 'This looks bad.'",
]

for prompt in test_prompts:
    print(f"\n📥 Input: {prompt}")
    response = generate_response(prompt)
    print(f"📤 Output: {response}")
    print("-" * 60)

In [ ]:
# Test with new examples (generalization)
print("\n📝 Testing with NEW examples (not in training data):")
print("=" * 60)

new_prompts = [
    "Rewrite to be more polite: 'Why haven't you responded yet?'",
    "Shorten: 'I would like to take this opportunity to express my sincere gratitude.'",
    "Make less formal: 'We kindly request your presence at the meeting.'",
    "Convert to bullets: 'First we plan, then we execute, finally we review.'",
]

for prompt in new_prompts:
    print(f"\n📥 Input: {prompt}")
    response = generate_response(prompt)
    print(f"📤 Output: {response}")
    print("-" * 60)

## Step 10: Download the Model (Optional)

Download your fine-tuned model to use locally or upload to Hugging Face Hub.

In [ ]:
# Zip the model for easy download
!zip -r phi3-text-rewriter-final.zip ./phi3-text-rewriter-final

print("\n✅ Model zipped! Click the folder icon on the left to download.")
print("   File: phi3-text-rewriter-final.zip")

## Step 11: Load and Use the Saved Model Later

Here's how to load your fine-tuned model in a new session:

In [ ]:
# Code to load the model in a new session
load_model_code = '''
# Load the fine-tuned model in a new session
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch

# Base model
model_id = "microsoft/Phi-3-mini-4k-instruct"
adapter_path = "./phi3-text-rewriter-final"  # Path to your saved adapters

# Quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Load LoRA adapters
model = PeftModel.from_pretrained(base_model, adapter_path)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(adapter_path)

print("Model loaded and ready!")
'''

print("📋 Code to load your model in a new session:")
print("=" * 50)
print(load_model_code)

## 🎉 Congratulations!

You've successfully fine-tuned Phi-3 Mini for text rewriting tasks!

### Key Takeaways:

1. **QLoRA** allows fine-tuning large models on limited GPU memory by using 4-bit quantization
2. **LoRA adapters** train only ~0.1% of total parameters, making training fast and efficient
3. **Chat formatting** is crucial - always use the model's expected prompt template
4. **Small datasets can work** - even 47 examples can teach the model new behaviors

### Tips for Better Results:

- **More data**: Add more diverse examples (aim for 100-1000+ for best results)
- **Data quality**: Ensure examples are high-quality and consistent
- **Longer training**: Try 5-10 epochs for small datasets
- **Hyperparameter tuning**: Experiment with learning rate, LoRA rank, and batch size

### Next Steps:

1. Upload to [Hugging Face Hub](https://huggingface.co/) to share your model
2. Merge LoRA weights into base model for faster inference
3. Convert to GGUF format for use with llama.cpp or Ollama